# Trình thu thập dữ liệu môi trường (VnExpress Spotlight)

Notebook này dùng để tải dữ liệu mới nhất từ các nguồn đo lường quốc gia:
1. Dữ liệu mực nước hồ chứa (Thủy Lợi Việt Nam)
2. Cảnh báo sạt lở & lũ quét (NCHMF)
3. Mực nước sông (VNDMS)

> **Hướng dẫn:** Chạy ô "Cài đặt" đầu tiên, sau đó chọn ô bạn cần và chạy để tải dữ liệu về máy.

In [ ]:
# === 1. Cài đặt và chuẩn bị (Chạy đầu tiên) ===
!git clone https://github.com/lqtue/environmental-data-hub.git 2>/dev/null || echo "Repo đã tồn tại"
%cd environmental-data-hub
!pip install -q pandas requests bs4 python-dateutil

import sys
from pathlib import Path
sys.path.insert(0, str(Path().absolute()))
from google.colab import files
print("✅ Môi trường đã sẵn sàng!")

In [ ]:
# === 2. Xuất dữ liệu nước hồ ===
# Điền ngày bắt đầu và kết thúc (YYYY-MM-DD), hoặc để trống để lấy theo mặc định.
start_date = "2025-10-07" # @param {type:"string"}
end_date = "" # @param {type:"string"}

from crawlers.lake_water import crawl
from datetime import datetime
import pytz

out_file = Path("lake.csv")
end = end_date if end_date else datetime.now(pytz.timezone('Asia/Ho_Chi_Minh')).strftime('%Y-%m-%d')
crawl(start_date, end, out_file)

if out_file.exists():
    files.download(str(out_file))

In [ ]:
# === 3. Xuất dữ liệu sạt lở lũ quét ===
mode = "refresh" # @param ["refresh", "historical"]
hist_start = "2025-11-06 00:00" # @param {type:"string"}
hist_end = "" # @param {type:"string"}

from crawlers.landslide import run_historical, run_refresh
from datetime import datetime
import pytz

out_file = Path("landslide.csv")
if mode == "refresh":
    run_refresh(out_file)
else:
    end = hist_end if hist_end else datetime.now(pytz.timezone('Asia/Ho_Chi_Minh')).replace(minute=0, second=0).strftime('%Y-%m-%d %H:%M')
    run_historical(hist_start, end, out_file)

if out_file.exists():
    files.download(str(out_file))

In [ ]:
# === 4. Xuất dữ liệu mực nước sông ===
days_history = "7" # @param {type:"string"}

from crawlers.river_water import crawl
from crawlers.config import ALLOWED_STATION_IDS

out_file = Path("river_long.csv")
crawl(days_history, out_file, ALLOWED_STATION_IDS)

if out_file.exists():
    files.download(str(out_file))